In [1]:
CostCurve = list[float]

def roi_model(dev_cost, cost_per_task, maint_per_month, value_per_task,
              tasks_per_month, ramp_months=0, horizon=24):
    monthly_cost = maint_per_month + cost_per_task * tasks_per_month
    cumulative, curve, payback = -dev_cost, [], None
    for month in range(1, horizon + 1):                                    #A
        maturity = min(month / ramp_months, 1.0) if ramp_months else 1.0
        cumulative += maturity * value_per_task * tasks_per_month - monthly_cost
        curve.append(cumulative)
        if payback is None and cumulative >= 0:
            payback = month
    steady_net = value_per_task * tasks_per_month - monthly_cost
    return steady_net / monthly_cost, payback, curve

tasks = 50_000                                                             #B

fine_tune = roi_model(dev_cost=150_000, cost_per_task=0.005, maint_per_month=3_000,
                      value_per_task=0.30, tasks_per_month=tasks)          #C
rag = roi_model(dev_cost=100_000, cost_per_task=0.02, maint_per_month=5_000,
                value_per_task=0.45, tasks_per_month=tasks, ramp_months=3)
agent = roi_model(dev_cost=250_000, cost_per_task=0.08, maint_per_month=10_000,
                  value_per_task=0.90, tasks_per_month=tasks, ramp_months=9)

for name, (monthly_return, payback, curve) in [("Fine-tuned model", fine_tune),
                                               ("RAG system", rag),
                                               ("RAG-enhanced agent", agent)]:
    print(f"{name}: monthly return={monthly_return:.2f}, "
          f"payback={payback} months, cumulative at 24 months=${curve[-1]:,.0f}")

#A Walk month by month so the ramp and the cumulative curve fall out of the same loop
#B Tasks per month, held constant across the three designs
#C Value per task differs because each design completes a different share of the work

Fine-tuned model: monthly return=3.62, payback=13 months, cumulative at 24 months=$132,000
RAG system: monthly return=2.75, payback=8 months, cumulative at 24 months=$273,500
RAG-enhanced agent: monthly return=2.21, payback=14 months, cumulative at 24 months=$314,000
